# Preprocessing Digikala Reviews

In [2]:
import csv
import re
import gc

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from hazm import (
    word_tokenize,
    stopwords_list,
    Normalizer,
    InformalNormalizer,
    Lemmatizer
)

## 1. Load Dataset

The Digikala reviews dataset contains several text fields, including the review title, main review body, advantages, and disadvantages.

For sentiment analysis, these text fields are combined into a single text input, while `recommendation_status` is retained as the target label.

In [ ]:
pd.set_option('display.max_colwidth', None)
file_path = "../data/digikala-comments.csv"

target_df = pd.read_csv(file_path, usecols=['title','body','advantages','disadvantages','recommendation_status'])

target_df.head()

,title,body,recommendation_status,advantages,disadvantages
0,پیشنهاد نمیشود,به درد نمیخوره,not_recommended,NaN,NaN
1,بسته بندی بد,می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بندی بشه,recommended,['تجربه جالبی بود برام '],['بسته بندی جالبی نداشت']
2,برس ریمل,بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خوبه من برای روغن زدم به مژه و لیفت ابرو خریدم,recommended,NaN,NaN
3,خوبه و خوشرنگ,به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد من برای صابون ابرو استفاده میکنم راضی ام,recommended,NaN,NaN
4,برس رنگ مو,معمولیه اگه واسه خونه رنگ کردن شخصی میخواین اوکیه ولی اگه واسه کار دکلره میخواین نه زیاد ولی خوبه,recommended,NaN,NaN


## 2. Merge Text Columns

The `title`, `body`, `advantages`, and `disadvantages` fields are combined into a single text column.

Missing values are replaced with empty strings before concatenation.

In [12]:
text_columns = ["title","body","advantages","disadvantages"]
target_df["body"] = (target_df[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip())
target_df = target_df[["body", "recommendation_status"]]

target_df.head()

,body,recommendation_status
0,پیشنهاد نمیشود به درد نمیخوره,not_recommended
1,بسته بندی بد می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بندی بشه ['تجربه جالبی بود برام '] ['بسته بندی جالبی نداشت'],recommended
2,برس ریمل بسته بندیش خوب بود\r\n کاربرد و کیفیتشم خیلی خوبه من برای روغن زدم به مژه و لیفت ابرو خریدم,recommended
3,خوبه و خوشرنگ به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد من برای صابون ابرو استفاده میکنم راضی ام,recommended
4,برس رنگ مو معمولیه اگه واسه خونه رنگ کردن شخصی میخواین اوکیه ولی اگه واسه کار دکلره میخواین نه زیاد ولی خوبه,recommended


## 3. Remove Empty Texts and Missing Labels

Reviews with empty text provide no useful information for text-based classification and are therefore removed.

Rows with missing `recommendation_status` are also removed because a target label is required for supervised learning.

In [13]:
print("Before removing empty texts and missing labels:", len(target_df))
target_df = target_df[target_df["body"].str.strip().ne("")]
target_df = target_df.dropna(subset=["recommendation_status"])
print("After removing empty texts and missing labels:", len(target_df))

Before removing empty texts and missing labels: 6156289
After removing empty texts and missing labels: 5261853


## 4. Remove Duplicate Reviews

Duplicate reviews are removed based on the combined review text.

Removing duplicates helps prevent repeated reviews from disproportionately affecting the model during training.

In [14]:
print("Before removing duplicates:", len(target_df))
target_df = target_df.drop_duplicates(subset=["body"])
print("\nAfter removing duplicates:", len(target_df))

Before removing duplicates: 5261853

After removing duplicates: 4048254


## 5. Class Distribution

The distribution of `recommendation_status` is examined to identify potential class imbalance in the dataset.

The three target classes are `recommended`, `not_recommended`, and `no_idea`.

In [16]:
label_counts = target_df['recommendation_status'].value_counts()
label_counts

,count
recommendation_status,
recommended,3086001
no_idea,495668
not_recommended,466585


## 6. Handle Class Imbalance

The `recommended` class contains substantially more samples than the other classes.

To reduce the class imbalance, the `recommended` class is downsampled to a maximum of 1,000,000 samples, while all samples from the `no_idea` and `not_recommended` classes are retained.

A fixed random seed is used to make the sampling reproducible.

In [17]:
recommended_df = target_df[target_df["recommendation_status"] == "recommended"]

other_df = target_df[target_df["recommendation_status"] != "recommended"]

recommended_sample = recommended_df.sample(n=1_000_000,random_state=42)

target_df = pd.concat([recommended_sample, other_df],ignore_index=True)

# Release unused DataFrames to reduce memory usage
del recommended_df, other_df, recommended_sample
gc.collect()

print(len(target_df))
print(target_df["recommendation_status"].value_counts())

1962253
recommendation_status
recommended        1000000
no_idea             495668
not_recommended     466585
Name: count, dtype: int64


## 7. Label Encoding

The categorical target labels are converted into numerical class IDs for model training.

The mapping is defined as follows:

- `not_recommended` → `0`
- `no_idea` → `1`
- `recommended` → `2`

In [18]:
label_mapping = {"not_recommended": 0,"no_idea": 1,"recommended": 2}

target_df["recommendation_status"] = (target_df["recommendation_status"].map(label_mapping))

target_df["recommendation_status"].value_counts()

,count
recommendation_status,
2,1000000
1,495668
0,466585


## 8. Stopword Analysis

Hazm provides a predefined list of Persian stopwords. Before removing stopwords, the list is inspected to identify words that may still carry useful sentiment or descriptive information.

Some words are later excluded from the stopword list to preserve their potential contribution to sentiment classification.

In [19]:
stop_words = stopwords_list()

print("Number of stopwords:", len(stop_words))
print(stop_words)

Number of stopwords: 389
['آخرین', 'آقای', 'آمد', 'آمده', 'آمده_است', 'آن', 'آنان', 'آنجا', 'آنها', 'آنچه', 'آنکه', 'آورد', 'آوری', 'آیا', 'ابتدا', 'اثر', 'اجرا', 'اخیر', 'از', 'است', 'اش', 'اغلب', 'افراد', 'افرادی', 'افزود', 'البته', 'اما', 'امر', 'امکان', 'اند', 'او', 'اول', 'اولین', 'اکنون', 'اگر', 'ایشان', 'این', 'اینجا', 'اینکه', 'با', 'بار', 'باره', 'باز', 'باشد', 'باشند', 'باعث', 'بالا', 'باید', 'بخش', 'بخشی', 'بدون', 'بر', 'برابر', 'براساس', 'برای', 'برخی', 'برداری', 'بروز', 'بزرگ', 'بسیار', 'بسیاری', 'بعد', 'بعضی', 'بلکه', 'بنابراین', 'بندی', 'به', 'بهتر', 'بهترین', 'بود', 'بودن', 'بودند', 'بوده', 'بوده_است', 'بی', 'بیان', 'بیرون', 'بیش', 'بیشتر', 'بیشتری', 'بین', 'تا', 'تاکنون', 'تبدیل', 'تحت', 'ترتیب', 'تعداد', 'تعیین', 'تغییر', 'تمام', 'تمامی', 'تنها', 'تهیه', 'تو', 'جا', 'جاری', 'جای', 'جایی', 'جدی', 'جدید', 'جریان', 'جز', 'جمع', 'جمعی', 'حال', 'حالا', 'حالی', 'حتی', 'حد', 'حداقل', 'حدود', 'حل', 'خاص', 'خاطرنشان', 'خصوص', 'خطر', 'خواهد_بود', 'خواهد_شد', 'خواهد_کرد', 'خوب',

### Customize the Stopword List

Not all predefined stopwords are removed from the reviews.

Some descriptive, sentiment-related, and negation words are preserved because removing them may result in the loss of useful information for sentiment classification.

The final stopword list is obtained by removing these selected words from Hazm's default stopword list.


In [21]:
descriptive_words = {'بزرگ', 'بسیار','بسیاری', 'بهتر', 'بهترین','بیش', 'بیشتر', 'بیشتری', 'حل', 'خاص', 'خوب', 'خوبی', 'خیلی', 'زیاد', 'زیادی', 'شروع', 'عالی',
                     'لازم', 'متفاوت', 'متاسفانه', 'مشخص', 'مناسب', 'مهم', 'نظیر', 'نیاز', 'پیدا', 'کافی', 'کامل', 'کاملا', 'کم', 'کمی', 'کند',
                     'اثر', 'بالا', 'بسیاری', 'تغییر', 'خطر', 'رشد', 'زیاد', 'زیادی', 'کافی', 'کامل', 'کاملا', 'کوچک', 'آخرین','ابتدا','اجرا',
                     'جدی','جدید','نظر','کلی','اغلب','عالی','عالیه','نباید','نبود','ندارد','نیست','نمی‌شود'}

stop_words = set(stopwords_list()) - descriptive_words

## 9. Text Preprocessing

The review texts are processed through several steps to prepare them for sentiment classification:

1. Remove non-Persian characters and normalize whitespace.
2. Normalize Persian text using Hazm.
3. Normalize informal words.
4. Select the most appropriate normalized form when multiple alternatives are generated.
5. Lemmatize the words while preserving negation.
6. Remove stopwords after lemmatization.
7. Join the processed tokens into the final review text.

In [22]:
normalizer = Normalizer()
informal_normalizer = InformalNormalizer()
lemmatizer = Lemmatizer()

def clean_and_process(text):
    # Remove non-Persian characters and normalize whitespace
    text = re.sub(r'[^آ-ی\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Normalize Persian text
    text = normalizer.normalize(text)

    # Normalize informal words and generate possible alternatives
    parsed = informal_normalizer.normalize(text)  # [[[opt1, opt2, ...], ...]]

    final_words = []

    for sentence in parsed:
        for word_options in sentence:
            # Select the most appropriate normalized form
            if len(word_options) > 2:
                chosen = None

                for w in word_options:
                    if w.endswith(' است') and re.match(r'^(.*[^ه])[\s\u200c]است$', w):
                        chosen = w
                        break
                if chosen is None:
                    chosen = word_options[1]
            else:
                chosen = word_options[0]

            # Process each token separately if the selected form contains multiple words
            for token in chosen.split():
                if not token.strip():
                    continue

                # Lemmatize while preserving the Persian negation prefix "ن"
                if re.match(r'^ن[^ن]', token):
                    lemma = 'ن' + lemmatizer.lemmatize(token[1:])
                else:
                    lemma = lemmatizer.lemmatize(token)

                 # Remove additional information returned by the lemmatizer
                lemma_clean = lemma.split('#')[0] if '#' in lemma else lemma

                 # Remove stopwords after lemmatization
                if lemma_clean not in stop_words:
                    final_words.append(lemma_clean)

    # Join the processed tokens into the final review text
    return ' '.join(final_words)

comments = target_df["body"].apply(clean_and_process)

print(comments[:10])

0                     ازش راضی داخل شگفت‌انگیز خرید عالی هرک بوش فهمید خوشش
1                             بافت اسفنج کاملاً نرم خوب خوب بلند خیسشم بهتر
2                                   شامپو ضد شوره شامپو راضی ضعیف ترشو خرید
3    بسیار شیک خاص طرفدار همیشگی به‌شخصه دریا باشگاه هرجا پوشید هرنظری عالی
4                                                            محافظ خوبی هست
5                                                       بازی بسیار خوب عالی
6                                        معمولی صرفه قیمت قیمت بازار معمولی
7                                                           برنامه‌ریزی خوب
8                                                             خوب چندتا خوب
9                                                            راضی خوشگل خوب
Name: body, dtype: object


## 10. Create and Save Final Dataset

The processed reviews and their numerical labels are combined into the final dataset.

The resulting dataset contains two columns:

- `comments`: Preprocessed review text
- `label`: Numerical target label

The final dataset is saved as a CSV file for use in the subsequent modeling stages.

In [ ]:
data = pd.DataFrame({'comments': comments,'label': target_df['recommendation_status']})

file_path = "../data/digikala_preprocessed.csv"

data.to_csv(file_path,index=False,encoding="utf-8")

print(f"File saved to {file_path}")

File saved to /content/drive/MyDrive/Colab Notebooks/digikala_preprocessed.csv


### Final Check

The final dataset is inspected to verify its shape and structure before being used for model training.

In [24]:
print("Final dataset shape:", data.shape)
data.head()

Final dataset shape: (1962253, 2)


,comments,label
0,ازش راضی داخل شگفت‌انگیز خرید عالی هرک بوش فهمید خوشش,2
1,بافت اسفنج کاملاً نرم خوب خوب بلند خیسشم بهتر,2
2,شامپو ضد شوره شامپو راضی ضعیف ترشو خرید,2
3,بسیار شیک خاص طرفدار همیشگی به‌شخصه دریا باشگاه هرجا پوشید هرنظری عالی,2
4,محافظ خوبی هست,2
